In [1]:
# ============================================================
# NB1 — COMPLETE STANDALONE: Feature Extraction + Bug-Fixed Interventions
# Dataset: nazmusresan/fitzpatrick17k (downloaded automatically via kaggle API)
#
# WHAT THIS DOES
# --------------
# 1. Downloads Fitzpatrick17k from Kaggle (needs your kaggle.json)
# 2. Extracts features from all 4 backbones (CLIP, ViT, ResNet50, DINOv2)
#    → saves to features/ so re-runs skip extraction
# 3. Runs the bug-fixed intervention loop (dark pool held out BEFORE test mask)
# 4. Prints comparison vs original published numbers
# 5. Saves results to results_bugfix/
#
# SETUP (Kaggle notebook: nothing needed, dataset auto-attached)
# LOCAL SETUP:
#   pip install kaggle transformers torchvision timm scikit-learn imbalanced-learn tqdm scipy pandas pillow
#   Place kaggle.json in ~/.kaggle/kaggle.json (chmod 600)
#   OR set KAGGLE_USERNAME / KAGGLE_KEY env vars
#   Then run: python nb1_complete_standalone.py
#
# GPU: strongly recommended for feature extraction (~1.5h on T4, ~6h on CPU)
# CPU: works fine for the probe/intervention loop (sklearn only)
# ============================================================

# ── INSTALL (uncomment on Kaggle or Colab) ───────────────────
# !pip install transformers timm scikit-learn imbalanced-learn tqdm scipy kaggle -q

import os
import sys
import warnings
import json
import shutil
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
from imblearn.over_sampling import SMOTE
from scipy.stats import fisher_exact
from tqdm import tqdm
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════
# CONFIG — edit paths here if needed
# ══════════════════════════════════════════════════════════════
CFG = dict(
    # ── Paths ──────────────────────────────────────────────────
    # Kaggle: these are auto-set. Local: point at your CSV + image dir.
    FITZ_CSV     = None,   # auto-detected; override with e.g. "data/fitzpatrick17k.csv"
    IMG_DIR      = None,   # auto-detected; override with e.g. "data/images/"
    FEATURES_DIR = "features",
    RESULTS_DIR  = "results_bugfix",

    # ── Models ─────────────────────────────────────────────────
    CLASS_LABELS = ["non-neoplastic", "benign", "malignant"],
    MODELS       = ["clip", "vit", "resnet50", "dinov2"],
    MODEL_LABELS = {"clip": "CLIP ViT-L/14", "vit": "ViT-B/16",
                    "resnet50": "ResNet-50", "dinov2": "DINOv2-Base"},
    MODEL_IDS    = {
        "clip":    "openai/clip-vit-large-patch14",
        "vit":     "google/vit-base-patch16-224",
        "resnet50": "torchvision",
        "dinov2":  "facebook/dinov2-base",
    },

    # ── Extraction ─────────────────────────────────────────────
    BATCH_SIZE   = 32,

    # ── Probe / Intervention ───────────────────────────────────
    SEEDS        = [42, 0, 1, 7, 99],
    N_BOOTSTRAP  = 1000,

    REAL_OVERSAMPLE_N = 200,
    GDRO_LR           = 1e-4,
    GDRO_EPOCHS       = 20,
    GDRO_BATCH_SIZE   = 64,
    GDRO_ETA          = 0.01,
    ADV_LR            = 1e-3,
    ADV_EPOCHS        = 50,
    ADV_LAMBDA        = 1.0,

    MAD_GAP_THRESH = 20.0,
    MAD_ACC_THRESH = 0.05,
)

for d in [CFG['FEATURES_DIR'], CFG['RESULTS_DIR']]:
    os.makedirs(d, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


# ══════════════════════════════════════════════════════════════
# SECTION 0A — DATASET DOWNLOAD (auto-skipped if already present)
# ══════════════════════════════════════════════════════════════
def find_fitzpatrick_files():
    """
    Try to locate the CSV and image directory.
    Checks common locations: Kaggle input paths, local 'data/', current dir.
    Returns (csv_path, img_dir) or (None, None) if not found.
    """
    candidates_csv = [
        # Kaggle dataset paths (various slug variants seen in the wild)
        "/kaggle/input/fitzpatrick17k/fitzpatrick17k.csv",
        "/kaggle/input/fitzpatrick17k/fitzpatrick17k (1).csv",
        "/kaggle/input/datasets/fitzpatrick17k/fitzpatrick17k.csv",
        "/kaggle/input/datasets/nazmusresan/fitzpatrick17k/New folder/fitzpatrick17k (1).csv",
        "/kaggle/input/fitzpatrick17k-dataset/fitzpatrick17k.csv",
        # Local
        "data/fitzpatrick17k.csv",
        "fitzpatrick17k.csv",
    ]
    candidates_img = [
        "/kaggle/input/fitzpatrick17k/images",
        "/kaggle/input/fitzpatrick17k/background removed",
        "/kaggle/input/datasets/nazmusresan/fitzpatrick17k/New folder/background removed",
        "/kaggle/input/fitzpatrick17k-dataset/images",
        "data/images",
        "images",
    ]

    csv_path = None
    img_dir  = None

    # Try overrides first
    if CFG['FITZ_CSV'] and os.path.exists(CFG['FITZ_CSV']):
        csv_path = CFG['FITZ_CSV']
    else:
        for c in candidates_csv:
            if os.path.exists(c):
                csv_path = c
                break

    if CFG['IMG_DIR'] and os.path.exists(CFG['IMG_DIR']):
        img_dir = CFG['IMG_DIR']
    else:
        for c in candidates_img:
            if os.path.exists(c):
                img_dir = c
                break

    return csv_path, img_dir


def download_fitzpatrick_kaggle():
    """Download via kaggle API. Requires kaggle.json or env vars."""
    print("\n── Attempting Kaggle download ──")
    try:
        import kaggle  # noqa: F401
    except ImportError:
        print("kaggle package not installed. Run: pip install kaggle")
        return False
    try:
        import subprocess
        out = subprocess.run(
            ["kaggle", "datasets", "download",
             "-d", "nazmusresan/fitzpatrick17k",
             "--unzip", "-p", "data/"],
            capture_output=True, text=True
        )
        print(out.stdout[-500:] if out.stdout else "")
        if out.returncode != 0:
            print("Kaggle download error:", out.stderr[-300:])
            return False
        print("✓ Kaggle download complete.")
        return True
    except Exception as e:
        print(f"Kaggle download failed: {e}")
        return False


def load_dataframe(csv_path, img_dir):
    df = pd.read_csv(csv_path)
    df = df[df['fitzpatrick_scale'].notna() & (df['fitzpatrick_scale'] > 0)]
    df = df[df['three_partition_label'].isin(CFG['CLASS_LABELS'])]

    # Build image file lookup
    image_files = {}
    for root, _, files in os.walk(img_dir):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                stem = f.rsplit('.', 1)[0]
                image_files[stem] = os.path.join(root, f)

    df['local_path'] = df['md5hash'].map(image_files)
    df = df[df['local_path'].notna()].copy()

    class_map = {name: i for i, name in enumerate(CFG['CLASS_LABELS'])}
    df['target'] = df['three_partition_label'].map(class_map)
    df['fitzpatrick_scale'] = df['fitzpatrick_scale'].astype(int)
    print(f"✓ Loaded {len(df)} valid images.")
    return df


# Try to find existing files; download if missing
csv_path, img_dir = find_fitzpatrick_files()
if csv_path is None or img_dir is None:
    print("Dataset not found locally. Trying Kaggle download...")
    success = download_fitzpatrick_kaggle()
    if success:
        csv_path, img_dir = find_fitzpatrick_files()
        # After download, try common unzipped paths
        if csv_path is None:
            for root, dirs, files in os.walk("data"):
                for f in files:
                    if f.endswith('.csv') and 'fitzpatrick' in f.lower():
                        csv_path = os.path.join(root, f)
                        break
        if img_dir is None:
            for root, dirs, files in os.walk("data"):
                n_imgs = sum(1 for f in files if f.lower().endswith(('.jpg','.png','.jpeg')))
                if n_imgs > 1000:
                    img_dir = root
                    break

if csv_path is None or img_dir is None:
    print("\n" + "="*60)
    print("ERROR: Could not find Fitzpatrick17k dataset.")
    print("="*60)
    print("""
Please do one of the following:

OPTION 1 — Kaggle (recommended):
  Set KAGGLE_USERNAME and KAGGLE_KEY env vars, or place kaggle.json at ~/.kaggle/kaggle.json
  Then re-run this script.

OPTION 2 — Manual download:
  1. Go to: https://www.kaggle.com/datasets/nazmusresan/fitzpatrick17k
  2. Download and extract to data/
  3. Set CFG['FITZ_CSV'] and CFG['IMG_DIR'] at the top of this script.

OPTION 3 — You have cached features/ from a previous nb_mega run:
  Just set SKIP_EXTRACTION = True below and the script will skip to the probe loop.
""")
    sys.exit(1)

print(f"CSV:     {csv_path}")
print(f"Images:  {img_dir}")
df = load_dataframe(csv_path, img_dir)


# ══════════════════════════════════════════════════════════════
# SECTION 0B — FEATURE EXTRACTION (skips if .npy files exist)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("SECTION 0: FEATURE EXTRACTION")
print("="*60)


def get_backbone(model_name):
    mid = CFG['MODEL_IDS'][model_name]
    if model_name == "clip":
        from transformers import CLIPModel, CLIPProcessor
        m = CLIPModel.from_pretrained(mid).to(device).eval()
        p = CLIPProcessor.from_pretrained(mid)
        return m, p
    elif model_name == "vit":
        from transformers import ViTModel, ViTImageProcessor
        m = ViTModel.from_pretrained(mid).to(device).eval()
        p = ViTImageProcessor.from_pretrained(mid)
        return m, p
    elif model_name == "resnet50":
        import torchvision.models as tv_models
        from torchvision import transforms
        m = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Identity()
        p = transforms.Compose([
            transforms.Resize(256), transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        return m.to(device).eval(), p
    elif model_name == "dinov2":
        from transformers import AutoModel, AutoImageProcessor
        m = AutoModel.from_pretrained(mid).to(device).eval()
        p = AutoImageProcessor.from_pretrained(mid)
        return m, p


@torch.no_grad()
def extract_split(dataframe, model_name, split_name):
    feat_file   = os.path.join(CFG['FEATURES_DIR'], f"{model_name}_{split_name}.npy")
    label_file  = os.path.join(CFG['FEATURES_DIR'], f"labels_{split_name}.npy")
    fitz_file   = os.path.join(CFG['FEATURES_DIR'], f"fitz_{split_name}.npy")

    if os.path.exists(feat_file):
        print(f"  ✓ Cached: {model_name} {split_name}")
        return

    print(f"\n  Extracting {model_name.upper()} | {split_name} | n={len(dataframe)}")
    backbone, proc = get_backbone(model_name)

    paths  = dataframe['local_path'].values
    labels = dataframe['target'].values.astype(int)
    fitz   = dataframe['fitzpatrick_scale'].values.astype(int)

    all_feats = []
    for i in tqdm(range(0, len(paths), CFG['BATCH_SIZE']), desc=f"  {model_name}/{split_name}"):
        batch_paths = paths[i:i+CFG['BATCH_SIZE']]
        images = []
        for p in batch_paths:
            try:
                images.append(Image.open(p).convert('RGB'))
            except Exception:
                images.append(Image.new('RGB', (224, 224)))

        if model_name == "resnet50":
            tensors = torch.stack([proc(img) for img in images]).to(device)
            feats = backbone(tensors)
        elif model_name == "clip":
            inputs = proc(images=images, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            vision_out = backbone.vision_model(**inputs)
            feats = backbone.visual_projection(vision_out.pooler_output)
        else:  # vit / dinov2
            inputs = proc(images=images, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            out = backbone(**inputs)
            feats = out.last_hidden_state[:, 0, :]

        feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-8)
        all_feats.append(feats.cpu().numpy())

    final_feats = np.vstack(all_feats)
    np.save(feat_file, final_feats)

    # Labels/fitz saved once (same for all models since same dataframe)
    if not os.path.exists(label_file):
        np.save(label_file, labels)
    if not os.path.exists(fitz_file):
        np.save(fitz_file, fitz)

    del backbone, proc
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


rand_df = df.copy()
demo_df = df.copy()

for m in CFG['MODELS']:
    extract_split(rand_df, m, "random")
    extract_split(demo_df, m, "demo")

print("\n✓ Feature extraction complete.")


# ══════════════════════════════════════════════════════════════
# SECTION 1 — HELPERS
# ══════════════════════════════════════════════════════════════
def load_features(model_name, split):
    fd = CFG['FEATURES_DIR']
    feats  = np.load(os.path.join(fd, f"{model_name}_{split}.npy"))
    labels = np.load(os.path.join(fd, f"labels_{split}.npy"))
    fitz   = np.load(os.path.join(fd, f"fitz_{split}.npy"))
    return feats, labels, fitz


def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float('nan'), float('nan'))
    p      = k / n
    denom  = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * (p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5 / denom
    return max(0.0, center - margin), min(1.0, center + margin)


def evaluate_full(y_true, y_proba, y_pred, fitz_arr=None):
    res = {}
    for c, name in [(0, 'non_neo'), (1, 'benign'), (2, 'malignant')]:
        m = (y_true == c)
        res[f'acc_{name}_dark'] = float((y_pred[m] == c).mean()) if m.sum() > 0 else float('nan')
    res['n_dark_benign'] = int((y_true == 1).sum())
    res['n_dark_total']  = len(y_true)
    n_b = int((y_true == 1).sum())
    k_b = int(((y_pred == 1) & (y_true == 1)).sum())
    res['fisher_p_benign'] = float('nan')
    if n_b > 0:
        try:
            expected = n_b / 3
            _, p = fisher_exact([[k_b, n_b - k_b],
                                  [int(expected), n_b - int(expected)]])
            res['fisher_p_benign'] = p
        except Exception:
            pass
    return res


# ══════════════════════════════════════════════════════════════
# SECTION 2 — DRO + ADVERSARIAL IMPLEMENTATIONS
# ══════════════════════════════════════════════════════════════
class GroupDROLinear(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.linear = nn.Linear(in_dim, n_classes)
    def forward(self, x):
        return self.linear(x)


def run_group_dro(train_f, train_y, train_groups, test_f, test_y, seed):
    torch.manual_seed(seed)
    n_classes = len(np.unique(train_y))
    n_groups  = len(np.unique(train_groups))
    model     = GroupDROLinear(train_f.shape[1], n_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG['GDRO_LR'])
    criterion = nn.CrossEntropyLoss(reduction='none')
    X = torch.tensor(train_f, dtype=torch.float32)
    Y = torch.tensor(train_y, dtype=torch.long)
    G = torch.tensor(train_groups, dtype=torch.long)
    group_weights = torch.ones(n_groups, device=device) / n_groups
    weight_log = []
    model.train()
    for ep in range(CFG['GDRO_EPOCHS']):
        perm = torch.randperm(len(X))
        X, Y, G = X[perm], Y[perm], G[perm]
        for i in range(0, len(X), CFG['GDRO_BATCH_SIZE']):
            xb = X[i:i+CFG['GDRO_BATCH_SIZE']].to(device)
            yb = Y[i:i+CFG['GDRO_BATCH_SIZE']].to(device)
            gb = G[i:i+CFG['GDRO_BATCH_SIZE']].to(device)
            logits = model(xb)
            losses = criterion(logits, yb)
            group_losses = torch.zeros(n_groups, device=device)
            for g in range(n_groups):
                mask = (gb == g)
                if mask.sum() > 0:
                    group_losses[g] = losses[mask].mean()
            group_weights = group_weights * torch.exp(CFG['GDRO_ETA'] * group_losses.detach())
            group_weights = group_weights / group_weights.sum()
            weighted_loss = (group_weights * group_losses).sum()
            optimizer.zero_grad()
            weighted_loss.backward()
            optimizer.step()
        weight_log.append(group_weights.cpu().detach().numpy().tolist())
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(test_f, dtype=torch.float32).to(device))
        proba  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = proba.argmax(axis=1)
    return proba, preds, weight_log


class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


class AdvDebiasModel(nn.Module):
    def __init__(self, in_dim, n_classes, n_groups):
        super().__init__()
        self.classifier = nn.Linear(in_dim, n_classes)
        self.adversary  = nn.Linear(in_dim, n_groups)
    def forward(self, x, lambd=1.0):
        logits = self.classifier(x)
        rev    = GradientReversal.apply(x, lambd)
        adv    = self.adversary(rev)
        return logits, adv


def run_adversarial_debiasing(train_f, train_y, train_groups, test_f, test_y, seed):
    torch.manual_seed(seed)
    n_classes = len(np.unique(train_y))
    n_groups  = max(2, len(np.unique(train_groups)))
    model     = AdvDebiasModel(train_f.shape[1], n_classes, n_groups).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG['ADV_LR'])
    criterion = nn.CrossEntropyLoss()
    X = torch.tensor(train_f, dtype=torch.float32).to(device)
    Y = torch.tensor(train_y, dtype=torch.long).to(device)
    G = torch.tensor(train_groups, dtype=torch.long).to(device)
    model.train()
    for ep in range(CFG['ADV_EPOCHS']):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), CFG['GDRO_BATCH_SIZE']):
            idx = perm[i:i+CFG['GDRO_BATCH_SIZE']]
            xb, yb, gb = X[idx], Y[idx], G[idx]
            optimizer.zero_grad()
            cls_logits, adv_logits = model(xb, lambd=CFG['ADV_LAMBDA'])
            loss = criterion(cls_logits, yb) + criterion(adv_logits, gb)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        logits, _ = model(torch.tensor(test_f, dtype=torch.float32).to(device))
        proba = torch.softmax(logits, dim=1).cpu().numpy()
        preds = proba.argmax(axis=1)
    return proba, preds


def apply_smote(train_f, train_y, seed):
    try:
        k = max(1, min(5, int(np.bincount(train_y).min()) - 1))
        sf, sy = SMOTE(random_state=seed, k_neighbors=k).fit_resample(train_f, train_y)
        norms = np.linalg.norm(sf, axis=1, keepdims=True)
        sf = sf / np.maximum(norms, 1e-8)
        return sf, sy
    except Exception as e:
        print(f"    SMOTE failed: {e}")
        return train_f, train_y


def eval_clf(train_f, train_y, test_f, test_y, test_fitz,
             rand_feats, rand_labels, seed, model_name, intervention,
             proba=None, preds=None, baseline_sgg=None):
    if proba is None:
        clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
        clf.fit(train_f, train_y)
        proba = clf.predict_proba(test_f)
        preds = clf.predict(test_f)

    demo_auc = roc_auc_score(test_y, proba, multi_class="ovr")
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    tr_idx, te_idx = next(sss.split(rand_feats, rand_labels))
    clf_r = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf_r.fit(rand_feats[tr_idx], rand_labels[tr_idx])
    rand_proba = clf_r.predict_proba(rand_feats[te_idx])
    rand_auc   = roc_auc_score(rand_labels[te_idx], rand_proba, multi_class="ovr")

    sgg     = rand_auc - demo_auc
    ref_sgg = baseline_sgg if baseline_sgg is not None else sgg
    gap_pct = max(0, (ref_sgg - sgg) / ref_sgg * 100) if ref_sgg > 0 else 0.0

    res = evaluate_full(test_y, proba, preds, test_fitz)
    benign_acc = res.get("acc_benign_dark", np.nan)
    n_b = int(res["n_dark_benign"])
    k_b = int(round(benign_acc * n_b)) if not np.isnan(benign_acc) else 0
    ci_lo, ci_hi = wilson_ci(k_b, n_b)

    mad_flag = ((baseline_sgg is not None and sgg > baseline_sgg * 1.2) and
                (benign_acc < CFG['MAD_ACC_THRESH'] if not np.isnan(benign_acc) else True))
    return {
        "model": model_name, "intervention": intervention, "seed": seed,
        "rand_auc": rand_auc, "demo_auc": demo_auc,
        "sgg": sgg, "gap_closed_pct": gap_pct,
        "acc_non_neo_dark":   res.get("acc_non_neo_dark", np.nan),
        "acc_benign_dark":    benign_acc,
        "acc_malignant_dark": res.get("acc_malignant_dark", np.nan),
        "benign_wilson_lo": ci_lo, "benign_wilson_hi": ci_hi,
        "n_dark_benign": n_b, "n_dark_total": int(res["n_dark_total"]),
        "fisher_p_benign": res.get("fisher_p_benign", np.nan),
        "mad_flag": mad_flag,
    }


# ══════════════════════════════════════════════════════════════
# SECTION 3 — BASELINE (unchanged from original nb_mega)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("SECTION 1: BASELINE (train light, test dark, no intervention)")
print("="*60)

baseline_rows = []
for model_name in CFG['MODELS']:
    print(f"\n── {model_name.upper()} ──")
    rand_feats, rand_labels, rand_fitz = load_features(model_name, "random")
    demo_feats, demo_labels, demo_fitz = load_features(model_name, "demo")

    for seed in tqdm(CFG['SEEDS'], desc=f"{model_name}"):
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
        tr_idx, te_idx = next(sss.split(rand_feats, rand_labels))
        clf_r = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
        clf_r.fit(rand_feats[tr_idx], rand_labels[tr_idx])
        rand_auc = roc_auc_score(rand_labels[te_idx],
                                  clf_r.predict_proba(rand_feats[te_idx]),
                                  multi_class="ovr")

        light_tr = np.isin(demo_fitz, [1, 2])
        dark_te  = np.isin(demo_fitz, [5, 6])
        clf_d = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
        clf_d.fit(demo_feats[light_tr], demo_labels[light_tr])
        demo_proba = clf_d.predict_proba(demo_feats[dark_te])
        demo_preds = clf_d.predict(demo_feats[dark_te])
        demo_auc   = roc_auc_score(demo_labels[dark_te], demo_proba, multi_class="ovr")
        demo_res   = evaluate_full(demo_labels[dark_te], demo_proba, demo_preds, demo_fitz[dark_te])

        sgg        = rand_auc - demo_auc
        benign_acc = demo_res.get("acc_benign_dark", np.nan)
        n_b        = int(demo_res["n_dark_benign"])
        k_b        = int(round(benign_acc * n_b)) if not np.isnan(benign_acc) else 0
        ci_lo, ci_hi = wilson_ci(k_b, n_b)

        baseline_rows.append({
            "model": model_name, "seed": seed,
            "rand_auc": rand_auc, "demo_auc": demo_auc, "sgg": sgg,
            "acc_non_neo_dark":   demo_res.get("acc_non_neo_dark", np.nan),
            "acc_benign_dark":    benign_acc,
            "acc_malignant_dark": demo_res.get("acc_malignant_dark", np.nan),
            "benign_wilson_lo":   ci_lo, "benign_wilson_hi": ci_hi,
            "n_dark_benign":      n_b,
            "n_dark_total":       int(demo_res["n_dark_total"]),
            "fisher_p_benign":    demo_res.get("fisher_p_benign", np.nan),
        })

df_base = pd.DataFrame(baseline_rows)
df_base.to_csv(os.path.join(CFG['RESULTS_DIR'], "00_baseline_results.csv"), index=False)

print("\n── Baseline summary (mean across seeds) ──")
print(f"{'Model':<14} {'Rand AUC':>9} {'Demo AUC':>9} {'SGG':>7} {'Benign%':>8}")
print("-" * 52)
for m in CFG['MODELS']:
    sub = df_base[df_base['model'] == m]
    print(f"{CFG['MODEL_LABELS'][m]:<14} "
          f"{sub['rand_auc'].mean():>9.3f} "
          f"{sub['demo_auc'].mean():>9.3f} "
          f"{sub['sgg'].mean():>7.3f} "
          f"{sub['acc_benign_dark'].mean()*100:>7.1f}%")

print("\n✓ Baseline saved.")


# ══════════════════════════════════════════════════════════════
# SECTION 4 — INTERVENTIONS (BUG-FIXED)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("SECTION 2: BUG-FIXED INTERVENTIONS")
print("  FIX: 200 dark images held out for pool BEFORE test mask is built,")
print("  so DRO/oversample/SMOTE/DAGC receive real dark training samples")
print("="*60)

intv_rows        = []
gdro_weight_logs = {}

for model_name in CFG['MODELS']:
    print(f"\n── {model_name.upper()} Interventions ──")
    rand_feats, rand_labels, rand_fitz = load_features(model_name, "random")
    demo_feats, demo_labels, demo_fitz = load_features(model_name, "demo")

    light_tr_mask = np.isin(demo_fitz, [1, 2])
    base_train_f  = demo_feats[light_tr_mask]
    base_train_y  = demo_labels[light_tr_mask]

    all_dark_idx = np.where(np.isin(demo_fitz, [5, 6]))[0]
    all_dark_y   = demo_labels[all_dark_idx]

    for seed in tqdm(CFG['SEEDS'], desc=f"{model_name} seeds"):
        rng = np.random.default_rng(seed)

        # ── THE BUG FIX ──────────────────────────────────────────
        # Stratified hold-out of ~200 dark images for the intervention pool
        # BEFORE building the test mask (original code: test mask = ALL dark → pool = empty)
        pool_local = []
        pool_cls_counts = {}
        for cls in [0, 1, 2]:
            cls_local = np.where(all_dark_y == cls)[0]
            n_take = min(
                int(round(CFG['REAL_OVERSAMPLE_N'] * len(cls_local) / len(all_dark_y))),
                len(cls_local)
            )
            chosen = rng.choice(cls_local, n_take, replace=False).tolist()
            pool_local.extend(chosen)
            pool_cls_counts[CFG['CLASS_LABELS'][cls]] = n_take
        pool_local = np.array(sorted(pool_local))
        test_local = np.array([i for i in range(len(all_dark_idx)) if i not in set(pool_local)])

        # Verify pool composition so stratification math can be audited
        if seed == CFG['SEEDS'][0]:  # print once per model (first seed only)
            total_pool = sum(pool_cls_counts.values())
            pool_str = ", ".join(
                f"{name}={n} ({n/max(total_pool,1)*100:.1f}%)"
                for name, n in pool_cls_counts.items()
            )
            print(f"  [seed={seed}] Pool composition: {pool_str} | total={total_pool} "
                  f"(dark total={len(all_dark_idx)}, test={len(test_local)})")

        pool_global = all_dark_idx[pool_local]
        test_global = all_dark_idx[test_local]

        dark_available_f = demo_feats[pool_global]
        dark_available_y = demo_labels[pool_global]
        dark_test_f      = demo_feats[test_global]
        dark_test_y      = demo_labels[test_global]
        dark_test_fitz   = demo_fitz[test_global]
        # ─────────────────────────────────────────────────────────

        # 1. Baseline
        baseline_row = eval_clf(
            base_train_f, base_train_y, dark_test_f, dark_test_y, dark_test_fitz,
            rand_feats, rand_labels, seed, model_name, "1_baseline", baseline_sgg=None
        )
        true_baseline_sgg = baseline_row['sgg']
        intv_rows.append(baseline_row)

        # 2. Real oversample (n=200)
        n_os    = min(CFG['REAL_OVERSAMPLE_N'], len(dark_available_f))
        idx_os  = rng.choice(len(dark_available_f), n_os, replace=True)
        over_f  = np.vstack([base_train_f, dark_available_f[idx_os]])
        over_y  = np.concatenate([base_train_y, dark_available_y[idx_os]])
        intv_rows.append(eval_clf(
            over_f, over_y, dark_test_f, dark_test_y, dark_test_fitz,
            rand_feats, rand_labels, seed, model_name, "2_real_oversample_200",
            baseline_sgg=true_baseline_sgg
        ))

        # 3. DAGC (oversample + demographic-aware group cost)
        n_base = len(base_train_y)
        n_add  = len(over_y) - n_base
        w = np.concatenate([
            np.full(n_base, (n_base + n_add) / (2 * n_base)),
            np.full(n_add,  (n_base + n_add) / (2 * max(n_add, 1))),
        ]) if n_add > 0 else None
        clf_dagc = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
        clf_dagc.fit(over_f, over_y, sample_weight=w)
        intv_rows.append(eval_clf(
            over_f, over_y, dark_test_f, dark_test_y, dark_test_fitz,
            rand_feats, rand_labels, seed, model_name, "3_real_oversample_DAGC",
            proba=clf_dagc.predict_proba(dark_test_f),
            preds=clf_dagc.predict(dark_test_f),
            baseline_sgg=true_baseline_sgg
        ))

        # 4. Group DRO (now with real dark samples — the key fix)
        n_dro   = min(CFG['REAL_OVERSAMPLE_N'], len(dark_available_f))
        idx_dro = rng.choice(len(dark_available_f), n_dro, replace=False)
        gdro_f  = np.vstack([base_train_f, dark_available_f[idx_dro]])
        gdro_y  = np.concatenate([base_train_y, dark_available_y[idx_dro]])
        groups  = np.concatenate([np.zeros(len(base_train_y), int), np.ones(n_dro, int)])
        # Guard: if pool happened to be all one class, groups could be all-zeros (no dark added)
        n_unique_groups = len(np.unique(groups))
        assert n_unique_groups == 2, (
            f"[seed={seed}, model={model_name}] train_groups has only {n_unique_groups} unique "
            f"value(s) — expected 2 (light=0, dark=1). "
            f"n_dro={n_dro}, dark pool size={len(dark_available_f)}. "
            f"This means DRO/adversarial debiasing received no dark training samples."
        )
        gdro_proba, gdro_preds, weight_log = run_group_dro(
            gdro_f, gdro_y, groups, dark_test_f, dark_test_y, seed)
        gdro_weight_logs[(model_name, seed)] = weight_log
        intv_rows.append(eval_clf(
            gdro_f, gdro_y, dark_test_f, dark_test_y, dark_test_fitz,
            rand_feats, rand_labels, seed, model_name, "4_group_dro",
            proba=gdro_proba, preds=gdro_preds,
            baseline_sgg=true_baseline_sgg
        ))

        # 5. Adversarial debiasing (now with real dark samples)
        adv_proba, adv_preds = run_adversarial_debiasing(
            gdro_f, gdro_y, groups, dark_test_f, dark_test_y, seed)
        intv_rows.append(eval_clf(
            gdro_f, gdro_y, dark_test_f, dark_test_y, dark_test_fitz,
            rand_feats, rand_labels, seed, model_name, "5_adversarial_debiasing",
            proba=adv_proba, preds=adv_preds,
            baseline_sgg=true_baseline_sgg
        ))

        # 6. SMOTE
        smote_f, smote_y = apply_smote(over_f, over_y, seed)
        intv_rows.append(eval_clf(
            smote_f, smote_y, dark_test_f, dark_test_y, dark_test_fitz,
            rand_feats, rand_labels, seed, model_name, "6_smote",
            baseline_sgg=true_baseline_sgg
        ))

df_intv = pd.DataFrame(intv_rows)
df_intv.to_csv(os.path.join(CFG['RESULTS_DIR'], "04_intervention_matrix.csv"), index=False)

# Save DRO weight trajectories
weight_logs_serializable = {f"{k[0]}_seed{k[1]}": v
                            for k, v in gdro_weight_logs.items()}
with open(os.path.join(CFG['RESULTS_DIR'], 'gdro_weight_trajectories.json'), 'w') as f:
    json.dump(weight_logs_serializable, f, indent=2)

print(f"\n✓ Bug-fixed intervention matrix saved → {CFG['RESULTS_DIR']}/04_intervention_matrix.csv")
print(f"✓ DRO weight trajectories saved → {CFG['RESULTS_DIR']}/gdro_weight_trajectories.json")


# ══════════════════════════════════════════════════════════════
# SECTION 5 — COMPARISON VS PUBLISHED + INTERPRETATION
# ══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("COMPARISON: Bug-fixed vs Published (Table S1)")
print("="*60)

PUBLISHED = {
    'clip': {
        '1_baseline':              {'demo_auc': 0.746, 'benign_acc': 0.079},
        '2_real_oversample_200':   {'demo_auc': 0.746, 'benign_acc': 0.079},
        '3_real_oversample_DAGC':  {'demo_auc': 0.746, 'benign_acc': 0.079},
        '4_group_dro':             {'demo_auc': 0.701, 'benign_acc': 0.000},
        '5_adversarial_debiasing': {'demo_auc': 0.577, 'benign_acc': 0.000},
        '6_smote':                 {'demo_auc': 0.737, 'benign_acc': 0.470},
    },
    'vit': {
        '1_baseline':              {'demo_auc': 0.695, 'benign_acc': 0.074},
        '4_group_dro':             {'demo_auc': 0.634, 'benign_acc': 0.000},
        '5_adversarial_debiasing': {'demo_auc': 0.575, 'benign_acc': 0.000},
        '6_smote':                 {'demo_auc': 0.684, 'benign_acc': 0.467},
    },
    'resnet50': {
        '1_baseline':              {'demo_auc': 0.676, 'benign_acc': 0.064},
        '4_group_dro':             {'demo_auc': 0.563, 'benign_acc': 0.000},
        '5_adversarial_debiasing': {'demo_auc': 0.499, 'benign_acc': 0.000},
        '6_smote':                 {'demo_auc': 0.670, 'benign_acc': 0.300},
    },
    'dinov2': {
        '1_baseline':              {'demo_auc': 0.721, 'benign_acc': 0.039},
        '4_group_dro':             {'demo_auc': 0.649, 'benign_acc': 0.000},
        '5_adversarial_debiasing': {'demo_auc': 0.542, 'benign_acc': 0.000},
        '6_smote':                 {'demo_auc': 0.715, 'benign_acc': 0.324},
    },
}

for model_name in CFG['MODELS']:
    print(f"\n{CFG['MODEL_LABELS'][model_name]}:")
    print(f"  {'Intervention':<28} {'Pub AUC':>8} {'New AUC':>8} {'Pub Ben%':>9} {'New Ben%':>9}")
    print(f"  {'-'*66}")
    sub = df_intv[df_intv['model'] == model_name]
    for intv in sorted(sub['intervention'].unique()):
        rows    = sub[sub['intervention'] == intv]
        new_auc = rows['demo_auc'].mean()
        new_ben = rows['acc_benign_dark'].mean()
        pub     = PUBLISHED.get(model_name, {}).get(intv, {})
        pub_auc = pub.get('demo_auc', float('nan'))
        pub_ben = pub.get('benign_acc', float('nan'))
        flag = " ← MAD" if rows['mad_flag'].any() else ""
        print(f"  {intv:<28} {pub_auc:>8.3f} {new_auc:>8.3f} "
              f"{pub_ben*100:>8.1f}% {new_ben*100:>8.1f}%{flag}")

# ── DRO weight collapse summary ───────────────────────────────
print("\n── Group DRO weight trajectories (minority weight by epoch) ──")
print(f"  {'Model':<14} {'Seed':>5}  ep1      ep5      ep10     ep20")
print(f"  {'-'*60}")
for (model_name, seed), log in sorted(gdro_weight_logs.items()):
    log_arr = np.array(log)
    # minority weight = log_arr[:, 1] if 2 groups
    if log_arr.shape[1] >= 2:
        mw = log_arr[:, 1]
        idxs = [0, 4, 9, 19]
        vals = [f"{mw[min(i, len(mw)-1)]:.3f}" for i in idxs]
        print(f"  {CFG['MODEL_LABELS'][model_name]:<14} {seed:>5}  {'  '.join(vals)}")

# ── Interpretation ────────────────────────────────────────────
print("\n" + "="*60)
print("INTERPRETATION GUIDE")
print("="*60)

dro_rows  = df_intv[df_intv['intervention'] == '4_group_dro']
smote_rows = df_intv[df_intv['intervention'] == '6_smote']
clip_dro_ben  = dro_rows[dro_rows['model'] == 'clip']['acc_benign_dark'].mean()
clip_smote_ben = smote_rows[smote_rows['model'] == 'clip']['acc_benign_dark'].mean()

print(f"\nCLIP Group DRO benign accuracy (bug-fixed): {clip_dro_ben*100:.1f}%")
print(f"CLIP SMOTE benign accuracy (bug-fixed):     {clip_smote_ben*100:.1f}%")
print()

if clip_dro_ben < 0.05:
    print("✅ DRO collapse is REAL (not a bug artifact).")
    print("   Even with proper dark training samples, DRO collapses benign accuracy to ~0%.")
    print("   This confirms the paper's central claim: the nc/Ng = 9.4% condition")
    print("   causes structural collapse regardless of implementation correctness.")
    print("   → Paper's MAD finding stands. Framing: 'We verified the bug-fixed version")
    print("     replicates identical collapse, confirming the result is structural.'")
elif clip_dro_ben < 0.15:
    print("⚠️  DRO shows partial collapse (not zero, not full recovery).")
    print("   → Revise collapse language to 'near-zero' and report exact numbers.")
else:
    print("❌ DRO benign accuracy recovered substantially after bug fix.")
    print("   The original collapse was a bug artifact, not a structural failure.")
    print("   → Major reframing needed. DRO failure under fine-tuning still holds.")

if clip_smote_ben > 0.35:
    print("\n✅ SMOTE recovery result holds.")
else:
    print(f"\n⚠️  SMOTE recovery lower than published ({clip_smote_ben*100:.1f}% vs 47.0%).")
    print("   This may reflect the smaller test set (pool withheld).")
    print("   Report both numbers and note the pool/test split difference.")

print("\n" + "="*60)
print("OUTPUT FILES")
print("="*60)
print(f"  {CFG['RESULTS_DIR']}/00_baseline_results.csv")
print(f"  {CFG['RESULTS_DIR']}/04_intervention_matrix.csv")
print(f"  {CFG['RESULTS_DIR']}/gdro_weight_trajectories.json")
print("\nPaste this entire output back to Claude for next steps.")

Device: cuda
CSV:     /kaggle/input/datasets/nazmusresan/fitzpatrick17k/New folder/fitzpatrick17k (1).csv
Images:  /kaggle/input/datasets/nazmusresan/fitzpatrick17k/New folder/background removed
✓ Loaded 16012 valid images.

SECTION 0: FEATURE EXTRACTION

  Extracting CLIP | random | n=16012


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...


── DINOV2 Interventions ──


dinov2 seeds:   0%|          | 0/5 [00:00<?, ?it/s]

  [seed=42] Pool composition: non-neoplastic=162 (81.0%), benign=19 (9.5%), malignant=19 (9.5%) | total=200 (dark total=2168, test=1968)


dinov2 seeds: 100%|██████████| 5/5 [05:49<00:00, 69.82s/it]


✓ Bug-fixed intervention matrix saved → results_bugfix/04_intervention_matrix.csv
✓ DRO weight trajectories saved → results_bugfix/gdro_weight_trajectories.json

COMPARISON: Bug-fixed vs Published (Table S1)

CLIP ViT-L/14:
  Intervention                  Pub AUC  New AUC  Pub Ben%  New Ben%
  ------------------------------------------------------------------
  1_baseline                      0.746    0.744      7.9%      7.9%
  2_real_oversample_200           0.746    0.754      7.9%      6.6%
  3_real_oversample_DAGC          0.746    0.761      7.9%      4.8%
  4_group_dro                     0.701    0.695      0.0%      0.0% ← MAD
  5_adversarial_debiasing         0.577    0.754      0.0%      8.7%
  6_smote                         0.737    0.744     47.0%     36.4%

ViT-B/16:
  Intervention                  Pub AUC  New AUC  Pub Ben%  New Ben%
  ------------------------------------------------------------------
  1_baseline                      0.695    0.695      7.4%      7.4%